# app

Aplikasi ini menggunakan Streamlit. Untuk membuka antarmuka, ekspor kode menjadi app.py dan jalankan `python -m streamlit run app.py`. Menjalankan sel saja tidak membuka antarmuka Streamlit.

BASE_DIR menyesuaikan direktori kerja notebook. Notebook belum dieksekusi.

In [ ]:
import os, re, pickle
import torch
import torch.nn as nn
import streamlit as st
from transformers import BertTokenizerFast, BertForSequenceClassification

BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
MAX_LEN = 30
BERT_MAX_LEN = 48
BERT_DIR = os.path.join(BASE_DIR, "bert_sarcasm_model")



In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, h):
        mu = torch.tanh(self.W(h))
        scores = self.v(mu).squeeze(-1)
        alpha = torch.softmax(scores, dim=-1)
        context = torch.sum(h * alpha.unsqueeze(-1), dim=1)
        return context, alpha

class HybridCNNBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, cnn_filters=64, lstm_hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv = nn.Conv1d(emb_dim, cnn_filters, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(emb_dim, lstm_hidden, bidirectional=True, batch_first=True)
        self.attn = AdditiveAttention(lstm_hidden * 2)
        self.fc1 = nn.Linear(cnn_filters + lstm_hidden * 2, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        e = self.emb(x)
        c = torch.relu(self.conv(e.transpose(1, 2)))
        c = torch.max(c, dim=2).values
        h, _ = self.lstm(e)
        ctx, alpha = self.attn(h)
        combined = torch.cat([c, ctx], dim=1)
        z = self.dropout(torch.relu(self.fc1(combined)))
        return self.fc2(z).squeeze(-1), alpha



In [ ]:
@st.cache_resource
def load_everything():
    with open(os.path.join(BASE_DIR, "vocab.pkl"), "rb") as f:
        vocab_data = pickle.load(f)
    word2idx = vocab_data["word2idx"]

    model_a = HybridCNNBiLSTMAttention(vocab_data["vocab_size"])
    model_a.load_state_dict(torch.load(os.path.join(BASE_DIR, "model_cnn_bilstm_attention.pt"), map_location="cpu"))
    model_a.eval()

    bert_tokenizer = BertTokenizerFast.from_pretrained(BERT_DIR)
    bert_model = BertForSequenceClassification.from_pretrained(BERT_DIR)
    bert_model.eval()

    metrics = None
    metrics_path = os.path.join(BASE_DIR, "metrics.pkl")
    if os.path.isfile(metrics_path):
        with open(metrics_path, "rb") as f:
            metrics = pickle.load(f)

    return word2idx, model_a, bert_tokenizer, bert_model, metrics

def clean(t):
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z' ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def encode(text, word2idx, max_len=MAX_LEN):
    ids = [word2idx.get(w, 1) for w in text.split()][:max_len]
    ids = ids + [0] * (max_len - len(ids))
    return torch.tensor([ids], dtype=torch.long)

def predict(text, word2idx, model_a, bert_tokenizer, bert_model):
    cleaned = clean(text)
    with torch.no_grad():
        logit_a, attn_weights = model_a(encode(cleaned, word2idx))
        prob_a = torch.sigmoid(logit_a).item()

        enc = bert_tokenizer(cleaned, truncation=True, max_length=BERT_MAX_LEN, return_tensors="pt")
        prob_bert = torch.softmax(bert_model(**enc).logits, dim=-1)[0, 1].item()

    tokens = cleaned.split()[:MAX_LEN]
    attn_scores = attn_weights[0][:len(tokens)].tolist()
    return cleaned, prob_a, prob_bert, tokens, attn_scores



In [ ]:
st.set_page_config(page_title="Deteksi Sarkasme Headline", page_icon="📰", layout="centered")
st.title("📰 Deteksi Sarkasme pada Judul Berita")
st.caption(
    "Membandingkan model **CNN-BiLSTM-Attention** (arsitektur paper Misra & Arora, AI Open 2023) "
    "dengan **BERT** (bert-base-uncased, fine-tuned) pada News Headlines Dataset."
)

try:
    word2idx, model_a, bert_tokenizer, bert_model, metrics = load_everything()
except (FileNotFoundError, OSError):
    st.error(
        "File model belum ditemukan (model_cnn_bilstm_attention.pt, vocab.pkl, dan folder bert_sarcasm_model).\n\n"
        "Jalankan dulu `python train_and_save.py` di folder yang sama sebelum menjalankan app ini."
    )
    st.stop()

with st.form(key="deteksi_form"):
    headline = st.text_input("Masukkan judul berita:", placeholder="Contoh: local man discovers water is wet")
    submitted = st.form_submit_button("Deteksi", type="primary")

if submitted and headline.strip():
    cleaned, prob_a, prob_bert, tokens, attn_scores = predict(headline, word2idx, model_a, bert_tokenizer, bert_model)

    col1, col2 = st.columns(2)
    for col, name, prob in [(col1, "CNN-BiLSTM-Attention", prob_a), (col2, "BERT", prob_bert)]:
        with col:
            st.subheader(name)
            label = "🟠 SARKASTIK" if prob > 0.5 else "🟢 BUKAN SARKASTIK"
            st.markdown(f"### {label}")
            st.progress(min(max(prob, 0.0), 1.0))
            st.caption(f"Skor keyakinan sarkastik: {prob*100:.1f}%")

    st.divider()
    st.markdown("**Kata yang paling diperhatikan model CNN-BiLSTM-Attention (attention weights):**")
    if tokens:
        max_score = max(attn_scores) if attn_scores else 1
        html = " ".join(
            f'<span style="background-color: rgba(255,140,0,{score/max_score:.2f}); '
            f'padding:2px 4px; border-radius:4px; margin:1px;">{tok}</span>'
            for tok, score in zip(tokens, attn_scores)
        )
        st.markdown(html, unsafe_allow_html=True)
        st.caption("Semakin gelap warna oranye = semakin besar bobot perhatian model pada kata tersebut.")

    with st.expander("Lihat teks setelah preprocessing"):
        st.code(cleaned)

if metrics:
    with st.sidebar:
        st.header("Performa Model (test set)")
        for name, key in [("CNN-BiLSTM-Attention", "cnn_bilstm_attention"), ("BERT", "bert")]:
            m = metrics[key]
            st.subheader(name)
            st.write(f"Accuracy: **{m['accuracy']*100:.1f}%**")
            st.write(f"Precision: **{m['precision']*100:.1f}%**")
            st.write(f"Recall: **{m['recall']*100:.1f}%**")
            st.write(f"F1-score: **{m['f1']*100:.1f}%**")
            st.markdown("---")